Se instala las librería google-generativeai para acceder a las APIs de generación de texto de Google.

In [ ]:
!pip install google-generativeai pandas

# Configuración de la API de Google Gemini

In [ ]:
import pandas as pd
import google.generativeai as genai
import os
import random

GOOGLE_API_KEY = ''

genai.configure(api_key=GOOGLE_API_KEY)

# Seleccionar el modelo de Gemini.
MODEL_NAME = 'models/gemini-1.5-flash'

print(f"API de Gemini configurada. Usando el modelo: {MODEL_NAME}")

# Carga y exploración inicial de los datasets


In [ ]:
import pandas as pd
import random

# Carga de los Datasets
try:
    df_sentiment = pd.read_csv("/content/dfSentimentAnalysis_cl_test.csv")
    df_hate = pd.read_csv("/content/dfHateSpeech_cl_test.csv")
    print("Datasets cargados exitosamente.")
    print(f"df_sentiment shape: {df_sentiment.shape}")
    print(f"df_hate shape: {df_hate.shape}")

    # primeras filas para verificar
    print("\nPrimeras filas de df_sentiment:")
    print(df_sentiment.head())
    print("\nPrimeras filas de df_hate:")
    print(df_hate.head())

except FileNotFoundError as e:
    print(f"Error al cargar el archivo: {e}")
    print("Por favor, sube los archivos CSV a tu entorno de Colab en el directorio /content/.")


# Para Sentiment Analysis
TWEET_COLUMN_SENTIMENT = 'text'
LABEL_COLUMN_SENTIMENT = 'labels'

SENTIMENT_LABELS_MAP = {
    0: 'negativo',
    1: 'positivo',
    2: 'neutral'
}

# Para Hate Speech
TWEET_COLUMN_HATE = 'text'
LABEL_COLUMN_HATE = 'labels'

HATE_LABELS_MAP = {
    0: 'no discurso de odio',
    1: 'discurso de odio'
}


print(f"\nClases únicas en '{LABEL_COLUMN_SENTIMENT}' (df_sentiment): {df_sentiment[LABEL_COLUMN_SENTIMENT].unique()}")
print(f"Clases únicas en '{LABEL_COLUMN_HATE}' (df_hate): {df_hate[LABEL_COLUMN_HATE].unique()}")

# Función para generar prompts One-shot

Esta función crea un prompt para la clasificación de un tweet, utilizando el enfoque One-shot. Es decir, incluye un solo ejemplo por cada clase de la tarea (sentimiento o discurso de odio) extraído aleatoriamente de un conjunto de ejemplos provisto.  
Si no se proporcionan ejemplos, genera un prompt simple sin ejemplos.  
El prompt resultante incluye el texto del tweet a clasificar y los ejemplos con sus etiquetas correspondientes para guiar al modelo en la tarea de clasificación.


In [ ]:
# Función para Generar Prompts (Solo One-shot)
# tweet: texto del tweet
# df_examples: el dataframe con ejemplos de tweets ya etiquetados
# task_type: odio o sentimiento
def create_prompt_one_shot(tweet, df_examples, task_type):

    # si el dataframe estaría vacío, entonces me devuelve un prompt directo para que el modelo clasifique sin contexto
    if df_examples.empty:
        # Aunque estamos en One-shot, si df_examples está vacío, no podemos dar un ejemplo.
        # En este caso, simplemente devolveremos una instrucción sin ejemplo.
        if task_type == 'sentiment':
            return f"""Clasificame según la clase de sentimiento (positivo/negativo/neutral) el siguiente tweet:
- Tweet: "{tweet}"
Sentimiento: """
        elif task_type == 'hate_speech':
            return f"""Clasificame según la clase de discurso de odio (discurso de odio/no discurso de odio) el siguiente tweet:
- Tweet: "{tweet}"
Clase: """
        else:
            raise ValueError("task_type debe ser 'sentiment' o 'hate_speech'")


    # Se eligen las clases, y el nombre de la columnas de los df
    examples_text = ""
    if task_type == 'sentiment':
        classes_keys = list(SENTIMENT_LABELS_MAP.keys())
        tweet_col = TWEET_COLUMN_SENTIMENT
        label_col = LABEL_COLUMN_SENTIMENT
    elif task_type == 'hate_speech':
        classes_keys = list(HATE_LABELS_MAP.keys())
        tweet_col = TWEET_COLUMN_HATE
        label_col = LABEL_COLUMN_HATE
    else:
        raise ValueError("task_type debe ser 'sentiment' o 'hate_speech'")

    ## Acá vamos a iterar por cada clase, para obtener un ejemplo por cada una para el one shot
    example_tweet_data = []
    for cls_key in classes_keys:
        class_tweets = df_examples[df_examples[label_col] == cls_key]
        if not class_tweets.empty:
            example_tweet = class_tweets.sample(n=1, random_state=random.randint(0, 10000)).iloc[0]

            label_text = SENTIMENT_LABELS_MAP[example_tweet[label_col]] if task_type == 'sentiment' else HATE_LABELS_MAP[example_tweet[label_col]]
            example_tweet_data.append(f"- Ejemplo: \"{example_tweet[tweet_col]}\": {label_text}")
        else:
            print(f"Advertencia: No hay ejemplos para la clase '{cls_key}' ({SENTIMENT_LABELS_MAP.get(cls_key, HATE_LABELS_MAP.get(cls_key, 'Desconocida'))}) para One-shot. Se elegirá un ejemplo genérico.")
            fallback_tweet = df_examples.sample(n=1, random_state=random.randint(0, 10000)).iloc[0]
            label_text_fallback = SENTIMENT_LABELS_MAP[fallback_tweet[label_col]] if task_type == 'sentiment' else HATE_LABELS_MAP[fallback_tweet[label_col]]
            example_tweet_data.append(f"- Ejemplo: \"{fallback_tweet[tweet_col]}\": {label_text_fallback} (Ejemplo genérico)")

    examples_text = "\n".join(example_tweet_data)

    if task_type == 'sentiment':
        return f"""En base a estos ejemplos:
{examples_text}
Clasificame según la clase de sentimiento (positivo/negativo/neutral) el siguiente tweet:
- Tweet: "{tweet}"
Sentimiento: """
    elif task_type == 'hate_speech':
        return f"""En base a estos ejemplos:
{examples_text}
Clasificame según la clase de discurso de odio (discurso de odio/no discurso de odio) el siguiente tweet:
- Tweet: "{tweet}"
Clase: """

print("Función para generar prompts de One-shot cargada.")

# Función para interactuar con la API de Gemini

Esta función envía un prompt al modelo de lenguaje Gemini utilizando la API oficial de Google Generative AI y retorna la respuesta generada.  



In [ ]:
# Función para interactuar con la API de Gemini

def get_gemini_classification(prompt, model_name=MODEL_NAME):
    """
    Envía el prompt a la API de Gemini y retorna la respuesta.
    """
    try:
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt,
            generation_config=genai.GenerationConfig(temperature=0.1),
            safety_settings={
                'HARM_CATEGORY_HARASSMENT': 'BLOCK_NONE',
                'HARM_CATEGORY_HATE_SPEECH': 'BLOCK_NONE',
                'HARM_CATEGORY_SEXUALLY_EXPLICIT': 'BLOCK_NONE',
                'HARM_CATEGORY_DANGEROUS_CONTENT': 'BLOCK_NONE',
            }
        )
        if hasattr(response, 'text'):
            return response.text.strip()
        elif response.candidates:
            return response.candidates[0].text.strip()
        else:
            return "No hay respuesta de texto disponible."
    except Exception as e:
        print(f"Error al llamar a la API de Gemini: {e}")
        return "ERROR_API"

print("Función get_gemini_classification cargada.")

# Ejemplo de Clasificación de Sentimiento usando One-shot Prompting

En esta celda se selecciona un tweet del dataset de análisis de sentimiento para clasificarlo utilizando un prompt de tipo one-shot.  
Se genera el prompt con un ejemplo de referencia y se envía a la función que interactúa con el modelo Gemini para obtener la predicción.  
Finalmente, se imprime tanto el prompt generado como la predicción resultante para ese tweet en particular.  
Esto permite validar el comportamiento del modelo en la tarea de clasificación de sentimiento con un solo ejemplo previo.


In [ ]:
print("\n--- EJEMPLO DE CLASIFICACIÓN DE SENTIMIENTO (ONE-SHOT) ---")
# Seleccionar un tweet del dataset de sentimiento para clasificar
# Puedo cambiar el índice para probar con diferentes tweets del df_sentiment, aca se usa el 5(indice)
tweet_to_classify_sentiment = df_sentiment.iloc[5][TWEET_COLUMN_SENTIMENT]

print("\n--- One-shot Sentiment ---")
prompt_one_sentiment = create_prompt_one_shot(tweet_to_classify_sentiment, df_sentiment, 'sentiment')
print("Prompt (One-shot):\n", prompt_one_sentiment)
prediction_one_sentiment = get_gemini_classification(prompt_one_sentiment)
print(f"Tweet a clasificar: \"{tweet_to_classify_sentiment}\"")
print(f"Predicción (One-shot): {prediction_one_sentiment}")
print("-" * 30)

# Ejemplo de Clasificación de Hate - Speech usando One-shot Prompting


In [ ]:

print("\n\n--- EJEMPLO DE CLASIFICACIÓN DE DISCURSO DE ODIO (ONE-SHOT) ---")
# Seleccionar un tweet del dataset de discurso de odio para clasificar
# Puedo cambiar el índice para probar con diferentes tweets del df, aca se usa el 5(indice)
tweet_to_classify_hate = df_hate.iloc[5][TWEET_COLUMN_HATE]

print("\n--- One-shot Hate Speech ---")
prompt_one_hate = create_prompt_one_shot(tweet_to_classify_hate, df_hate, 'hate_speech')
print("Prompt (One-shot):\n", prompt_one_hate)
prediction_one_hate = get_gemini_classification(prompt_one_hate)
print(f"Tweet a clasificar: \"{tweet_to_classify_hate}\"")
print(f"Predicción (One-shot): {prediction_one_hate}")
print("-" * 30)


# Clasificación Masiva de Tweets usando One-shot Prompting

Esta celda procesa todos los tweets de los datasets de análisis de sentimiento y discurso de odio.  
Para cada tweet, se genera un prompt one-shot con un ejemplo y se consulta al modelo Gemini para obtener la predicción.  
Se incluye una pausa de 6 segundos entre consultas para respetar límites de tasa de la API.  
Se muestra progreso parcial cada 5 tweets procesados y finalmente se imprimen las primeras filas con las predicciones agregadas.  


In [ ]:
import time

print("--- PROCESANDO TODOS LOS TWEETS CON ONE-SHOT ---")

# Procesar Sentiment Analysis
print("\nIniciando clasificación de Sentimiento (One-shot)...")
sentiment_predictions = []
for index, row in df_sentiment.iterrows():
    tweet_text = row[TWEET_COLUMN_SENTIMENT]

    prompt = create_prompt_one_shot(tweet_text, df_sentiment, 'sentiment')

    prediction = get_gemini_classification(prompt)
    sentiment_predictions.append(prediction)

    # pausa
    time.sleep(6)

    if (index + 1) % 5 == 0:
        print(f"  Procesado {index + 1}/{len(df_sentiment)} tweets de sentimiento.")

df_sentiment['gemini_one_shot_prediction'] = sentiment_predictions
print("Clasificación de Sentimiento completada.")
print("\nPrimeras filas del DataFrame de Sentimiento con predicciones:")
print(df_sentiment.head())

# --- Procesar Hate Speech ---
print("\nIniciando clasificación de Discurso de Odio (One-shot)...")
hate_predictions = []
for index, row in df_hate.iterrows():
    tweet_text = row[TWEET_COLUMN_HATE]

    prompt = create_prompt_one_shot(tweet_text, df_hate, 'hate_speech')
    prediction = get_gemini_classification(prompt)
    hate_predictions.append(prediction)

    # pausa
    time.sleep(6)

    if (index + 1) % 5 == 0:
        print(f"  Procesado {index + 1}/{len(df_hate)} tweets de discurso de odio.")

df_hate['gemini_one_shot_prediction'] = hate_predictions
print("Clasificación de Discurso de Odio completada.")
print("\nPrimeras filas del DataFrame de Discurso de Odio con predicciones:")
print(df_hate.head())


# Evaluación del Rendimiento del Modelo (Sentiment Analysis - One-shot)



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

print("\n--- EVALUANDO EL RENDIMIENTO DEL MODELO DE SENTIMIENTO ---")

# Conviertimos las etuqietas(0,1,2) a su nombre textual(negativo, positivo, neutral)
df_sentiment['true_label_text'] = df_sentiment[LABEL_COLUMN_SENTIMENT].map(SENTIMENT_LABELS_MAP)

# Función para limpiar y extraer la clase esperada de la predicción generada por Gemini
def extract_sentiment_label(prediction):
    prediction = str(prediction).lower().strip()
    if 'sentimiento:' in prediction:
        prediction = prediction.split('sentimiento:')[1].strip()

    if 'positivo' in prediction:
        return 'positivo'
    elif 'negativo' in prediction:
        return 'negativo'
    elif 'neutral' in prediction:
        return 'neutral'
    else:
        return 'UNK'  # desconocido o no clasificado claramente

df_sentiment['gemini_one_shot_prediction_cleaned'] = df_sentiment['gemini_one_shot_prediction'].apply(extract_sentiment_label)

# Filtrar filas con predicciones inválidas o errores
df_sentiment_filtered = df_sentiment[
    (df_sentiment['gemini_one_shot_prediction_cleaned'] != 'ERROR_API') &
    (df_sentiment['gemini_one_shot_prediction_cleaned'] != 'UNK')
]

true_labels_sentiment = df_sentiment_filtered['true_label_text']
predicted_labels_sentiment = df_sentiment_filtered['gemini_one_shot_prediction_cleaned']

print(f"\nNúmero de tweets evaluados: {len(true_labels_sentiment)} / {len(df_sentiment)}")
if not true_labels_sentiment.empty:
    print(f"Precisión (Accuracy): {accuracy_score(true_labels_sentiment, predicted_labels_sentiment):.4f}")

    print("\nReporte de Clasificación:")
    target_names_sentiment = ['negativo', 'neutral', 'positivo']
    print(classification_report(true_labels_sentiment, predicted_labels_sentiment,
                                target_names=target_names_sentiment, zero_division=0))

    print("\nMatriz de Confusión:")
    conf_matrix_sentiment = confusion_matrix(true_labels_sentiment, predicted_labels_sentiment,
                                             labels=target_names_sentiment)
    print(pd.DataFrame(conf_matrix_sentiment, index=target_names_sentiment, columns=target_names_sentiment))
else:
    print("No hay datos válidos para evaluar el sentimiento tras limpiar predicciones inválidas.")


# Evaluación del Rendimiento del Modelo (Discurso de Odio - One-shot)



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import re

print("\n--- EVALUANDO EL RENDIMIENTO DEL MODELO PARA DISCURSO DE ODIO ---")

# Mapear las etiquetas verdaderas a texto legible
df_hate['true_label_text'] = df_hate[LABEL_COLUMN_HATE].map(HATE_LABELS_MAP)

# Función para limpiar y normalizar las predicciones de Gemini para hate speech
def extract_hate_label(prediction):
    prediction = str(prediction).lower().strip()

    # Quitar negritas (**), saltos de línea, puntos y otros caracteres extra
    prediction = re.sub(r'\*\*', '', prediction)
    prediction = prediction.replace('\n', ' ').replace('.', ' ').strip()

    # Tomar solo la primera parte del texto para evitar ruido
    prediction_short = ' '.join(prediction.split()[:4])

    # Si aparece 'clase:', tomar lo que viene después
    if 'clase:' in prediction:
        prediction = prediction.split('clase:')[1].strip()

    if 'no discurso de odio' in prediction or 'no discurso de odio' in prediction_short:
        return 'no discurso de odio'
    elif 'discurso de odio' in prediction or 'discurso de odio' in prediction_short:
        return 'discurso de odio'
    else:
        return 'UNK'

# Aplicar la limpieza a las predicciones
df_hate['gemini_one_shot_prediction_cleaned'] = df_hate['gemini_one_shot_prediction'].apply(extract_hate_label)

# Filtrar filas con predicciones inválidas o errores
df_hate_filtered = df_hate[
    (df_hate['gemini_one_shot_prediction_cleaned'] != 'ERROR_API') &
    (df_hate['gemini_one_shot_prediction_cleaned'] != 'UNK')
]

true_labels_hate = df_hate_filtered['true_label_text']
predicted_labels_hate = df_hate_filtered['gemini_one_shot_prediction_cleaned']

print("\n--- Resultados para Discurso de Odio (One-shot) ---")
if not true_labels_hate.empty:
    print(f"Número de tweets de discurso de odio evaluados: {len(true_labels_hate)} / {len(df_hate)}")
    print(f"Precisión (Accuracy): {accuracy_score(true_labels_hate, predicted_labels_hate):.4f}")

    print("\nReporte de Clasificación:")
    target_names_hate = ['no discurso de odio', 'discurso de odio']
    print(classification_report(true_labels_hate, predicted_labels_hate, target_names=target_names_hate, zero_division=0))

    print("\nMatriz de Confusión:")
    conf_matrix_hate = confusion_matrix(true_labels_hate, predicted_labels_hate, labels=target_names_hate)
    print(pd.DataFrame(conf_matrix_hate, index=target_names_hate, columns=target_names_hate))
else:
    print("No hay datos válidos para evaluar discurso de odio después de limpiar errores o si todas las predicciones fueron 'ERROR_API'/'UNK'.")


In [ ]:
# Dataframes finales

print("\n--- Predicciones Limpias Discurso de Odio ---")
print(df_hate_filtered[['true_label_text', 'gemini_one_shot_prediction_cleaned']])

print("\n--- Predicciones Limpias Sentimiento ---")
print(df_sentiment_filtered[['true_label_text', 'gemini_one_shot_prediction_cleaned']])


# Gráficos

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix # Asegúrate de que confusion_matrix esté importada
from sklearn.preprocessing import LabelEncoder

def graficar_evaluacion(df, true_col, pred_col, target_names, titulo):
    true_labels = df[true_col]
    pred_labels = df[pred_col]

    le = LabelEncoder()

    le.classes_ = np.array(target_names, dtype=object)

    true_labels_encoded = le.transform(true_labels)
    pred_labels_encoded = le.transform(pred_labels)

    cm = confusion_matrix(true_labels_encoded, pred_labels_encoded, labels=np.arange(len(target_names)))
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Matriz de Confusión - {titulo}')
    plt.xlabel('Predicción')
    plt.ylabel('Etiqueta Verdadera')
    plt.show()


graficar_evaluacion(
    df_sentiment_filtered,
    true_col='true_label_text',
    pred_col='gemini_one_shot_prediction_cleaned',
    target_names=['negativo', 'neutral', 'positivo'],
    titulo='Análisis de Sentimiento'
)

graficar_evaluacion(
    df_hate_filtered,
    true_col='true_label_text',
    pred_col='gemini_one_shot_prediction_cleaned',
    target_names=['no discurso de odio', 'discurso de odio'],
    titulo='Discurso de Odio'
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Datos del Reporte de Clasificación de Sentimiento ---
sentiment_data = {
    'Class': ['negativo', 'neutral', 'positivo'],
    'Precision': [0.40, 0.60, 0.53],
    'Recall': [0.40, 0.30, 0.80],
    'F1-Score': [0.40, 0.40, 0.64],
}
df_sentiment_metrics = pd.DataFrame(sentiment_data)

# Datos del Reporte de Clasificación de Discurso de Odio ---
hate_data = {
    'Class': ['no discurso de odio', 'discurso de odio'],
    'Precision': [0.77, 0.71],
    'Recall': [0.67, 0.80],
    'F1-Score': [0.71, 0.75],
}
df_hate_metrics = pd.DataFrame(hate_data)

print("DataFrames de métricas creados:")
print("\nDataFrame Sentimiento:")
print(df_sentiment_metrics)
print("\nDataFrame Discurso de Odio:")
print(df_hate_metrics)

sentiment_csv_path = 'sentiment_metrics.csv'
hate_csv_path = 'hate_speech_metrics.csv'

df_sentiment_metrics.to_csv(sentiment_csv_path, index=False)
df_hate_metrics.to_csv(hate_csv_path, index=False)

print(f"\nMétricas de Sentimiento guardadas en: {sentiment_csv_path}")
print(f"Métricas de Discurso de Odio guardadas en: {hate_csv_path}")

# Función para Graficar
def plot_metrics_bars(df_metrics, title):
    classes = df_metrics['Class']
    metrics = ['Precision', 'Recall', 'F1-Score']
    x = np.arange(len(classes))
    width = 0.2
    plt.figure(figsize=(10, 6))

    for i, metric_name in enumerate(metrics):
        plt.bar(x + i * width, df_metrics[metric_name], width, label=metric_name)

    plt.xticks(x + width, classes, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.title(f'Métricas por Clase - {title}')
    plt.ylabel('Puntaje')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Generar los gráficos de barras ---
print("\nGenerando gráficos de barras")
plot_metrics_bars(df_sentiment_metrics, 'Análisis de Sentimiento')
plot_metrics_bars(df_hate_metrics, 'Discurso de Odio')
